In [1]:
import os
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint




c:\Users\Dev Singh\miniconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

TRAIN_DIR = "dataset/train"
VAL_DIR = "dataset/val"

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 30

MODEL_DIR = "model"
MODEL_PATH = os.path.join(MODEL_DIR, "fall3_cnn_detection.keras")

os.makedirs(MODEL_DIR, exist_ok=True)


# ============================================================
# 2. LOAD DATASET
# ============================================================

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels="inferred",
    label_mode="int",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ============================================================
# 3. CHECK CLASS NAMES
# ============================================================

class_names = train_dataset.class_names

print("\nClass names:")
print(class_names)

print("\nClass mapping:")
for index, class_name in enumerate(class_names):
    print(index, "->", class_name)


# ============================================================
# 4. PERFORMANCE OPTIMIZATION
# ============================================================

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.prefetch(
    buffer_size=AUTOTUNE
)

val_dataset = val_dataset.prefetch(
    buffer_size=AUTOTUNE
)


# ============================================================
# 5. DATA AUGMENTATION
# ============================================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10)
])




Found 374 files belonging to 2 classes.
Found 111 files belonging to 2 classes.

Class names:
['fallen', 'not_fallen']

Class mapping:
0 -> fallen
1 -> not_fallen


In [3]:
# ============================================================
# 6. BUILD CNN MODEL FROM SCRATCH
# ============================================================

model = models.Sequential([

    # Input
    layers.Input(shape=(150, 150, 3)),

    # Normalize pixel values
    layers.Rescaling(1.0 / 255),

    # Data augmentation
    data_augmentation,

    # --------------------------------------------------------
    # CNN BLOCK 1
    # --------------------------------------------------------

    layers.Conv2D(
        32,
        (3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling2D(
        (2, 2)
    ),

    # --------------------------------------------------------
    # CNN BLOCK 2
    # --------------------------------------------------------

    layers.Conv2D(
        64,
        (3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling2D(
        (2, 2)
    ),

    # --------------------------------------------------------
    # CNN BLOCK 3
    # --------------------------------------------------------

    layers.Conv2D(
        128,
        (3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling2D(
        (2, 2)
    ),

    # --------------------------------------------------------
    # CNN BLOCK 4
    # --------------------------------------------------------

    layers.Conv2D(
        256,
        (3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling2D(
        (2, 2)
    ),

    # --------------------------------------------------------
    # CLASSIFICATION
    # --------------------------------------------------------

    layers.Flatten(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.5),

    # Two classes:
    # 0 = fallen
    # 1 = not_fallen
    layers.Dense(
        2,
        activation="softmax"
    )
])


# ============================================================
# 7. DISPLAY MODEL ARCHITECTURE
# ============================================================

model.summary()


# ============================================================
# 8. COMPILE MODEL
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0001
    ),

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]
)


# ============================================================
# 9. CALLBACKS
# ============================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

model_checkpoint = ModelCheckpoint(
    MODEL_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)


# ============================================================
# 10. TRAIN MODEL
# ============================================================

print("\nStarting training...\n")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=[
        early_stopping,
        model_checkpoint
    ]
)




Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 150, 150, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 75, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 75, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 37, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 37, 37, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 18, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 18, 18, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 9, 9, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 20736)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,654,336 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,043,010 (11.61 MB)

 Trainable params: 3,043,010 (11.61 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...

Epoch 1/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.4700 - loss: 0.7138
Epoch 1: val_accuracy improved from None to 0.36937, saving model to model\fall3_cnn_detection.keras

Epoch 1: finished saving model to model\fall3_cnn_detection.keras
12/12 ━━━━━━━━━━━━━━━━━━━━ 9s 428ms/step - accuracy: 0.4385 - loss: 0.7181 - val_accuracy: 0.3694 - val_loss: 0.6989
Epoch 2/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 244ms/step - accuracy: 0.5405 - loss: 0.6889
Epoch 2: val_accuracy improved from 0.36937 to 0.63063, saving model to model\fall3_cnn_detection.keras

Epoch 2: finished saving model to model\fall3_cnn_detection.keras
12/12 ━━━━━━━━━━━━━━━━━━━━ 4s 311ms/step - accuracy: 0.5535 - loss: 0.6882 - val_accuracy: 0.6306 - val_loss: 0.6674
Epoch 3/30
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 243ms/step - accuracy: 0.5813 - loss: 0.6841
Epoch 3: val_accuracy did not improve from 0.63063
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 272ms/step - accuracy: 0.5668 - loss: 0.6885 - val_accuracy: 0.63

In [ ]:
# ============================================================
# 11. EVALUATE MODEL
# ============================================================

print("\nEvaluating model...\n")

val_loss, val_accuracy = model.evaluate(
    val_dataset,
    verbose=1
)

print("\n==============================")
print("FINAL VALIDATION RESULTS")
print("==============================")

print(
    f"Validation Loss     : {val_loss:.4f}"
)

print(
    f"Validation Accuracy : {val_accuracy * 100:.2f}%"
)


# ============================================================
# 12. SAVE FINAL MODEL
# ============================================================

model.save(MODEL_PATH)

print("\nModel saved successfully:")
print(MODEL_PATH)


# ============================================================
# 13. SAVE CLASS NAMES
# ============================================================

CLASS_FILE = os.path.join(
    MODEL_DIR,
    "class_names.txt"
)

with open(CLASS_FILE, "w") as file:

    for class_name in class_names:
        file.write(class_name + "\n")


print("Class names saved:")
print(CLASS_FILE)


# ============================================================
# 14. PLOT TRAINING / VALIDATION ACCURACY
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.title("Training vs Validation Accuracy")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.legend()

plt.grid(True)

plt.show()


# ============================================================
# 15. PLOT TRAINING / VALIDATION LOSS
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("Training vs Validation Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

plt.grid(True)

plt.show()


# ============================================================
# 16. FINAL INFORMATION
# ============================================================

print("\n======================================")
print("TRAINING COMPLETED")
print("======================================")

print("Classes:", class_names)

print("Image size:", IMG_SIZE)

print("Model:", MODEL_PATH)

print("Validation accuracy:",
      f"{val_accuracy * 100:.2f}%")

print("======================================")